**Статистическая языковая модель**

Евгений Борисов <esborisov@sevsu.ru>

подбираем наиболее вероятное продолжение цепочки слов

In [1]:
import gzip
import requests
from bs4 import BeautifulSoup

In [2]:
url='http://lib.ru/NEWPROZA/LOBAS/taxisty.txt'
text = BeautifulSoup(requests.get(url).text).get_text()
with gzip.open('taxisty.txt.gz','wt') as f: f.write(text)

# with gzip.open('../data/taxisty.txt.gz','rt') as f: text = f.read()

text = text[1030:-7261].strip() # выкидываем заголовок и хвост страницы 
print(f'символов:{len(text)}\n---------------\n'%())
print(text[:343])

символов:637782
---------------

ПРОЛОГ 
1. 
     Вы прилетели в  Нью-Йорк и  остановились  в  одном  из отелей, глядящих
окнами на Центральный парк.  Наутро по  приезде вы вышли из  отеля, вдохнули
полной грудью очищенный зеленью парка воздух  и,  взглянув на часы, --  пора
было начинать хлопотливый день, --  направились к  первому из  таксомоторов,
выстроившихся вереницей


In [1]:
# url='http://az.lib.ru/d/dostoewskij_f_m/text_0080.shtml'
# text = BeautifulSoup(requests.get(url).text).get_text()
# with gzip.open('dostoewskij.txt.gz','wt') as f: f.write(text)

# # with gzip.open('../data/dostoewskij.txt.gz','rt') as f: text = f.read()

# text = text[2876:-664184].strip() # выкидываем заголовок и хвост страницы 
# print(f'символов:{len(text)}\n---------------\n'%())
# print(text[:355])

In [4]:
from nltk import __version__ as nltk_version
print('nltk version:',nltk_version)

nltk version: 3.9.1


In [5]:
from tqdm.auto import tqdm
from random import sample

from nltk.tokenize import sent_tokenize as nltk_sentence_split
from nltk.tokenize import word_tokenize as nltk_tokenize_word

sentences = [ 
    nltk_tokenize_word(s,language='russian') # разбиваем предложения на слова
    for s in tqdm(nltk_sentence_split(text,language='russian')) # режем текст на отдельные предложения
]

print('предложений: %i\n'%(len(sentences)))
display( sample(sentences,1) )

  0%|          | 0/6651 [00:00<?, ?it/s]

предложений: 6651



[['Мальчишка',
  'давно',
  'отошел',
  'и',
  'курил',
  'на',
  'пару',
  'с',
  'другим',
  'пацаном',
  'самокрутку',
  'марихуаны',
  ';',
  'мы',
  'оба',
  '--',
  'и',
  'Доктор',
  ',',
  'и',
  'я',
  '--',
  'в',
  'равной',
  'степени',
  'были',
  'ему',
  'смешны',
  'и',
  'противны',
  '.']]

In [6]:
# sentences = sentences[:1024] # ограничиваем датасет для ускорения процеса 

---

In [7]:
# собираем словарь

from itertools import chain
from collections import Counter

text_tokens = list(chain(*sentences)) # собираем все токены (слова) из текста
vocab_size = len(set(text_tokens)) # размер словаря

text_tokens_freq = Counter( text_tokens ) # оценка частоты использования слов

print('всего слов тексте: %i'%(len(text_tokens)))
print('размер словаря: %i'%(vocab_size))
text_tokens_freq.most_common()[:10] # наиболее частые токены

всего слов тексте: 117085
размер словаря: 24067


[(',', 10727),
 ('--', 4221),
 ('.', 3999),
 ('и', 2453),
 ('в', 2407),
 ('не', 2077),
 ('я', 1513),
 ('!', 1470),
 ("''", 1412),
 ('...', 1392)]

----

In [8]:
# from nltk.util import bigrams
from nltk.util import ngrams as nltk_ngrams

# вынимаем все n-gram из текста
ngram_len = 2 # работаем с биграммами
text_ngrams = [ ngram for s in sentences for ngram in nltk_ngrams(s,ngram_len) ]
print('количество n-gram: %i'%(len(set(text_ngrams))))
sample(text_ngrams,5)

количество n-gram: 72677


[('из', 'старых'),
 ('Они', 'купили'),
 ('ее', 'шатало'),
 (',', 'углубилась'),
 ('из', 'узкого')]

In [9]:
# cчитаем частоту n-gram
text_ngrams_freq = Counter(text_ngrams)
sample( list(text_ngrams_freq.items()), 7)

[((',', 'подкрадывалась'), 1),
 (('Эф-Би-Ай37', 'и'), 1),
 (('смерзшихся', 'серых'), 1),
 (('будешь', '...'), 2),
 (('пассажирам', '--'), 1),
 (('--', 'приблизилась'), 1),
 (('второй', 'квотер'), 1)]

In [10]:
text_ngrams_freq.most_common()[:10] # наиболее частые n-ngram

[((',', 'что'), 807),
 ((',', 'и'), 631),
 ((',', 'а'), 466),
 ((',', '--'), 420),
 ((',', 'как'), 384),
 ((':', '--'), 366),
 ((',', 'но'), 319),
 ((',', 'я'), 297),
 (("''", ','), 294),
 ((',', 'не'), 247)]

----

Оценка вероятностей совместного использования слов со сглаживанием Лапласа


$$
P(w_n|w_{n-1}) = \frac{ C(w_{n-1} w_{n}) +1 }{ C(w_{n-1}) + V }
$$

In [14]:
text_ngrams_prob = { # оценка совместного использования токенов
    ngram : (text_ngrams_freq[ngram]+1) / ( text_tokens_freq[ ngram[0] ] + vocab_size )
    for ngram in text_ngrams_freq 
}

display( list(text_ngrams_prob.items())[:10] )

[(('ПРОЛОГ', '1'), 8.309788931361143e-05),
 (('1', '.'), 0.0009961813049975095),
 (('Вы', 'прилетели'), 0.0001243884235840451),
 (('прилетели', 'в'), 0.0001246416552411816),
 (('в', 'Нью-Йорк'), 0.0003399561834252474),
 (('Нью-Йорк', 'и'), 0.00016600954554886905),
 (('и', 'остановились'), 7.541478129713424e-05),
 (('остановились', 'в'), 8.308408109006314e-05),
 (('в', 'одном'), 0.0003021832741557755),
 (('одном', 'из'), 0.00029069767441860465)]

In [15]:
# оценка вероятности новых ngram по статистике text_ngrams_prob
def get_ngram_prob(ngram,text_ngrams_prob=text_ngrams_prob,text_tokens_freq=text_tokens_freq):
    if ngram in text_ngrams_prob: return text_ngrams_prob[ngram]
    token_0_freq = text_tokens_freq[ngram[0]] if ngram[0] in text_tokens_freq else 0.
    return 1./(token_0_freq+len(text_tokens_freq))

выбираем дополнение предложения с наибольшей вероятностью цепочки слов

$$
P(w_1,\ldots, w_n) = \prod_{k=1}^{n} P(w_k|w_k-1)
$$

In [16]:
from operator import mul
from functools import reduce 

# оценка предложения
def get_sentence_prob(
        sentence,
        text_ngrams_prob=text_ngrams_prob,
        text_tokens_freq=text_tokens_freq,
        ngram_len=ngram_len
    ):
    return reduce( mul, [ 
        get_ngram_prob(ngram,text_ngrams_prob=text_ngrams_prob,text_tokens_freq=text_tokens_freq)
        for ngram in nltk_ngrams(sentence,ngram_len) 
    ] )


In [17]:
# оценка всех возможных продолжений предложения
def get_next_token_prob(sentence,text_ngrams_prob=text_ngrams_prob,text_tokens_freq=text_tokens_freq):
    
    sentence_prob = get_sentence_prob( # оценка предложения
            sentence,
            text_ngrams_prob=text_ngrams_prob,
            text_tokens_freq=text_tokens_freq
        )
    
    token_next = { # оценки всех возможных продолжений
        ngram[1] : 
            sentence_prob*
              get_ngram_prob(ngram,text_ngrams_prob=text_ngrams_prob,text_tokens_freq=text_tokens_freq)
        for ngram in text_ngrams_prob if ngram[0]==sentence[-1] 
    }
    
    return token_next

In [18]:
from collections import OrderedDict

def get_top_prob_token(tokens_prob,n=3):  # n наиболее вероятных продолжений
    kf = lambda k: tokens_prob[k]
    return OrderedDict([
        (key,tokens_prob[key])
        for key in sorted(tokens_prob, key=kf, reverse=True)[:n]
    ])

----

In [20]:
# генерируем продолжения

for sentence in sample(sentences,10):
    if len(sentence)<10: continue
     # берём начало предложения
    sentence_ = sentence[:-(len(sentence)//4)]
    
    # генерируем возможные продолжения
    
    # считаем верояности продолжений
    next_token_prob = get_next_token_prob(sentence_) 
    
    # выбираем наиболее вероятные
    top_prob_token = get_top_prob_token(next_token_prob)
       
    print( 
        ' '.join(sentence_)
        +  ' ... ' 
        + '{ '
        + ' | '.join(top_prob_token.keys())
        + ' }'
        + '\n'
    )

-- И , не оборачиваясь , протянул на заднее сиденье ... { , | и | . }

-- Здесь похоронен Христофор Колумб , -- кивнул я , будто старому приятелю , на памятник великому мореплавателю , стоящий посреди ... { тротуара | дороги | улицы }

Испуганно вскрикнул пережатый стартер , я лихорадочно стал подавать назад , чтобы поскорее выбраться из-под моста , и -- ... { А | сказал | не }

Да так , ни о чем ... Странные отношения следователя с подследственным развивались ; вот они и болтали о всякой всячине ... Например , Гелий рассказывал свои тюремные сны ... Иногда грустные ... { , }

-- таксистам никто хорошо не платит ... И шепчутся , шепчутся ... И как мне знать : дом они ... { не | -- | и }

Это у меня получалось до того убедительно , что и жена , и сын всегда со мной соглашались ... Однако этот вредный кэбби , от которого я за десять минут услышал больше неприятных вещей , чем ... { не | я | ты }

Здесь , в Нью-Йорке , мы работаем совсем ... { не | уж | уже }

